# Sistema de recomendação
**Produto Alvo:** Motor de Popa 1949
**Mecanismo:** Filtro colaborativo (similaridade de cosseno)

In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

### 1 - carregamento e cruzamento dos dados

In [2]:
# carregar os datasets
orders = pd.read_csv('../lh_nautical_csv/orders.csv')
order_items = pd.read_csv('../lh_nautical_csv/order_items.csv')
product_variants = pd.read_csv('../lh_nautical_csv/product_variants.csv')
products = pd.read_csv('../lh_nautical_csv/products.csv')

# construir o dataframe de interações
df_interaction = pd.merge(order_items, orders, left_on='order_id', right_on='id')
df_interaction = pd.merge(df_interaction, product_variants, left_on='product_variant_id', right_on='id')

# manter apenas o ID do cliente e do produto
# remover duplicatas para focar apenas na presença ou ausencia
df_interaction = df_interaction[['customer_id', 'product_id']].drop_duplicates()

# criar flag de "comprou"
df_interaction['bought'] = 1

### 2 - matriz user-item e similaridade

In [3]:
# criar a matriz user-item
user_item_matrix = df_interaction.pivot(index='customer_id', columns='product_id', values='bought').fillna(0)

# calcular a similaridade de cosseno (producto x user)
item_item_sim_matrix = cosine_similarity(user_item_matrix.T)

# converter para DataFrame para buscas e cruzamentos
item_item_sim_df = pd.DataFrame(item_item_sim_matrix, index=user_item_matrix.columns, columns=user_item_matrix.columns)

### 3 - execução do ranking de recomendação

In [10]:
# identificar o produto alvo
target_name = 'Motor de Popa 1949'
target_product = products[products['name'] == target_name].iloc[0]
target_product_id = target_product['id']

# obter similaridades descartando o proprio item alvo e ordenando decrescente
similarities = item_item_sim_df[target_product_id].drop(target_product_id).sort_values(ascending=False)

# pegar os 5 primeiros
top_5 = similarities.head(5).reset_index()
top_5.columns = ['product_id', 'similarity']

# cruzar para obter os nomes legiveis e converter para porcentagem
top_5_with_names = pd.merge(top_5, products[['id', 'name']], left_on='product_id', right_on='id')
top_5_with_names['similarity'] = (top_5_with_names['similarity'] * 100).round(2)

# --- output ---
print("+" + "-"*60 + "+")
print("|" + " "*17 + "MOTOR DE RECOMENDACAO" + " "*22 + "|")
print("|" + "-"*60 + "|")
print(f"| ALVO: {target_name:<52} |")
print("+" + "-"*60 + "+")
print("| CLIENTES QUE COMPRARAM ISSO, TAMBEM LEVARAM:               |")
print("|                                                            |")
for index, row in top_5_with_names.iterrows():
    item_nome = f"{index + 1}. {row['name']}"
    sim_valor = f"[{row['similarity']}%]"
    print(f"| {item_nome:<46} {sim_valor:>11} |")
print("+" + "-"*60 + "+")

+------------------------------------------------------------+
|                 MOTOR DE RECOMENDACAO                      |
|------------------------------------------------------------|
| ALVO: Motor de Popa 1949                                   |
+------------------------------------------------------------+
| CLIENTES QUE COMPRARAM ISSO, TAMBEM LEVARAM:               |
|                                                            |
| 1. Motor de Popa 5331                             [25.66%] |
| 2. Cabo Náutico 2105                              [25.62%] |
| 3. Vela Mestra 1913                               [25.58%] |
| 4. Cabo Náutico 9048                              [23.93%] |
| 5. GPS Plotter 6249                               [23.77%] |
+------------------------------------------------------------+
